# IPL Model Training — Model A

This notebook trains classical ML models using the handcrafted features generated in `IPL_EDA_FeatureEngineering.ipynb`.

Models:
- Random Forest
- SVM + PCA
- XGBoost

Evaluation:
- Accuracy
- Precision (weighted)
- Recall (weighted)
- F1 Score (weighted)

Output:
- Best model selection
- Export as `model_<teamname>.pkl`


## 1. Import Libraries

In [6]:

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier

import joblib
from pathlib import Path


## 2. Load Features Dataset

In [3]:

DATA_DIR = Path("../data")

FEATURE_FILE = DATA_DIR / "features.csv"

df = pd.read_csv(FEATURE_FILE)
df = df.dropna()

print("Shape:", df.shape)
df.head()


/var/folders/1m/r53kmn99675gj6mlypwb9w9r0000gn/T/ipykernel_8921/2703622803.py:5: DtypeWarning: Columns (0: cell_id, 1: label, 2: f_1, 3: f_2, 4: f_3, 5: f_4, 6: f_5, 7: f_6, 8: f_7, 9: f_8, 10: f_9, 11: f_10, 12: f_11, 13: f_12, 14: f_13, 15: f_14, 16: f_15, 17: f_16, 18: f_17, 19: f_18, 20: f_19, 21: f_20, 22: f_21, 23: f_22, 24: f_23, 25: f_24, 26: f_25, 27: f_26, 28: f_27, 29: f_28, 30: f_29, 31: f_30, 32: f_31, 33: f_32, 34: f_33, 35: f_34, 36: f_35, 37: f_36, 38: f_37, 39: f_38, 40: f_39, 41: f_40, 42: f_41, 43: f_42, 44: f_43, 45: f_44, 46: f_45, 47: f_46, 48: f_47, 49: f_48, 50: f_49, 51: f_50, 52: f_51, 53: f_52, 54: f_53, 55: f_54, 56: f_55, 57: f_56, 58: f_57, 59: f_58, 60: f_59, 61: f_60, 62: f_61, 63: f_62, 64: f_63, 65: f_64, 66: f_65, 67: f_66, 68: f_67, 69: f_68, 70: f_69, 71: f_70, 72: f_71, 73: f_72, 74: f_73, 75: f_74, 76: f_75, 77: f_76, 78: f_77, 79: f_78, 80: f_79, 81: f_80, 82: f_81, 83: f_82, 84: f_83, 85: f_84, 86: f_85, 87: f_86, 88: f_87, 89: f_88, 90: f_89, 9

Shape: (448194, 3290)


,image_name,cell_id,label,f_1,f_2,f_3,f_4,f_5,f_6,f_7,...,f_3278,f_3279,f_3280,f_3281,f_3282,f_3283,f_3284,f_3285,f_3286,f_3287
0,GTvsLSG_image_0.jpg,1,0,0.819521,0.273765,0.267433,0.419384,0.057488,0.051157,0.010637,...,0.000000,0.253371,0.080312,0.046218,0.038778,0.057467,0.022716,0.101588,0.048187,0.177923
1,GTvsLSG_image_0.jpg,2,0,0.00727,0.100509,0.512661,0.542688,0.51519,0.195962,0.19533,...,0.322488,0.284265,0.035648,0.000000,0.000000,0.000000,0.000000,0.018844,0.322488,0.322488
2,GTvsLSG_image_0.jpg,3,0,0.0,0.042769,0.142231,0.323585,0.794706,0.330879,0.26888,...,0.020055,0.292465,0.038748,0.005562,0.008795,0.007866,0.063571,0.254887,0.337620,0.337620
3,GTvsLSG_image_0.jpg,4,0,0.093658,0.460096,0.391803,0.332096,0.277462,0.178731,0.353169,...,0.000000,0.037615,0.032329,0.166200,0.301281,0.301281,0.026942,0.008866,0.000000,0.000000
4,GTvsLSG_image_0.jpg,5,0,0.243183,0.84161,0.473506,0.079453,0.028934,0.02503,0.022504,...,0.026881,0.277008,0.277008,0.138843,0.137554,0.170011,0.091778,0.084151,0.099526,0.088811


## 3. Prepare Features and Labels

In [4]:

feature_cols = [c for c in df.columns if c.startswith("f_")]

X = df[feature_cols]
y = df["label"]

# Groups ensure all cells from the same image stay together
groups = df["image_name"]

print("Features:", X.shape)
print("Labels:", y.shape)
print("Images:", groups.nunique())


Features: (448194, 3287)
Labels: (448194,)
Images: 3606


## 4. Image-Aware Train/Test Split (GroupShuffleSplit)

In [5]:

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (358210, 3287)
Test : (89984, 3287)


## 5. Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)


ValueError: could not convert string to float: 'f_1'

: 

## 6. SVM with PCA

In [ ]:

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    ))
])

svm_pipeline.fit(X_train, y_train)

svm_pred = svm_pipeline.predict(X_test)


## 7. XGBoost

In [ ]:

num_classes = len(np.unique(y))

xgb_model = XGBClassifier(
    objective="multi:softmax",
    num_class=num_classes,
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)


## 8. Evaluation Function

In [ ]:

def evaluate_model(name, y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    rec = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print(f"\n{name}")
    print("-"*50)
    print("Accuracy :", round(acc,4))
    print("Precision:", round(prec,4))
    print("Recall   :", round(rec,4))
    print("F1 Score :", round(f1,4))

    return {
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    }


## 9. Compare Models

In [ ]:

results = []

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    )
)

results.append(
    evaluate_model(
        "SVM + PCA",
        y_test,
        svm_pred
    )
)

results.append(
    evaluate_model(
        "XGBoost",
        y_test,
        xgb_pred
    )
)

results_df = pd.DataFrame(results)
results_df.sort_values(
    by="F1",
    ascending=False
)


## 10. Select Best Model

In [ ]:

best_row = results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]

best_model_name = best_row["Model"]

if best_model_name == "Random Forest":
    best_model = rf_model
elif best_model_name == "SVM + PCA":
    best_model = svm_pipeline
else:
    best_model = xgb_model

print("Best Model:", best_model_name)
print(best_row)


## 11. Save Best Model

In [ ]:

TEAM_NAME = "teamname"   # Replace with your actual team name

model_file = f"model_{TEAM_NAME}.pkl"

joblib.dump(best_model, model_file)

print("Saved:", model_file)


## 12. Detailed Classification Report

In [ ]:

best_predictions = best_model.predict(X_test)

print(
    classification_report(
        y_test,
        best_predictions,
        zero_division=0
    )
)
